## Imports
Load libraries for modeling, evaluation, and plotting.

In [ ]:
import warnings                           # lets us hide unnecessary warnings
warnings.filterwarnings("ignore")         # suppresses warning messages for cleaner output

from pathlib import Path                  # helps work with file paths

import numpy as np                        # numerical operations
import pandas as pd                       # dataframe handling
import matplotlib.pyplot as plt           # plotting

from sklearn.linear_model import LogisticRegression      # simple classifier
from sklearn.ensemble import RandomForestClassifier      # tree-based classifier
from sklearn.metrics import accuracy_score, roc_auc_score
# two basic metrics for comparing models

from sklearn.preprocessing import StandardScaler         # rescales features
from sklearn.pipeline import Pipeline                    # chains preprocessing + model

## Load Processed Data
Read in the cleaned monthly dataset created by the data notebook.

In [ ]:
DATA_PATH = Path("data/copper_macro_model_data.csv")
# path to the processed model-ready dataset

df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
# load the csv into a dataframe
# index_col=0 uses the first column as the row index
# parse_dates=True converts the index into real dates

df = df.sort_index()
# make sure the rows are in time order

df.head()
# preview the first few rows

## Choose Features and Target
Define what the model uses as inputs and what it tries to predict.

In [ ]:
FEATURE_COLUMNS = [
    "vix",               # current monthly VIX level
    "vix_3m_avg",        # recent average VIX
    "vix_12m_z",         # how unusual VIX is relative to recent history
    "unrate",            # unemployment rate
    "cpi_yoy",           # inflation rate
    "fedfunds",          # interest rates
    "indpro_yoy",        # industrial production growth
    "dollar_yoy",        # dollar strength change
    "copper_ret_1m",     # last month's copper return
    "copper_mom_3m",     # 3-month copper momentum
    "copper_mom_6m"      # 6-month copper momentum
]
# these are the predictors the models will use

TARGET_COLUMN = "copper_next_up"
# 1 means copper goes up next month, 0 means it goes down

PRICE_COLUMN = "copper_price"
# current copper price

RETURN_TARGET_COLUMN = "copper_next_ret_1m"
# actual next-month return, used to estimate next-month price

## Prepare Modeling Table
Keep only the needed columns and remove missing values.

In [ ]:
model_df = df[FEATURE_COLUMNS + [TARGET_COLUMN, PRICE_COLUMN, RETURN_TARGET_COLUMN]].dropna().copy()
# keep only the columns needed for training and prediction
# drop rows with missing values
# copy() creates a separate dataframe for safety

model_df.head()
# preview the cleaned table

## Split Inputs and Target
Separate predictor variables from the target variable.

In [ ]:
X = model_df[FEATURE_COLUMNS]
# X holds the input features

y = model_df[TARGET_COLUMN]
# y holds the labels we want to predict

print("X shape:", X.shape)
# number of rows and columns in the feature matrix

print("y shape:", y.shape)
# number of target rows

## Train-Test Split
Train on older data and test on newer data.

In [ ]:
train_size = int(len(model_df) * 0.8)
# use 80% of the rows for training and the last 20% for testing

train_df = model_df.iloc[:train_size].copy()
# older rows become training data

test_df = model_df.iloc[train_size:].copy()
# newer rows become test data

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train period:", train_df.index.min(), "to", train_df.index.max())
print("Test period:", test_df.index.min(), "to", test_df.index.max())
# this confirms the time-based split

## Create Models
Set up two models and compare them.

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),                 # standardize the features
    ("model", LogisticRegression(max_iter=1000)) # logistic regression model
])
# a pipeline lets scaling and prediction happen together

rf_model = RandomForestClassifier(
    n_estimators=300,         # number of trees
    max_depth=4,              # keeps trees from getting too complex
    min_samples_leaf=3,       # prevents tiny leaf nodes
    random_state=42           # makes results reproducible
)
# random forest is a more flexible nonlinear model

## Train Models
Fit both models on the training data.

In [ ]:
logistic_model.fit(X_train, y_train)
# train the logistic regression model

rf_model.fit(X_train, y_train)
# train the random forest model

print("Both models trained.")

## Make Predictions
Generate predictions and probabilities on the test set.

In [ ]:
log_pred = logistic_model.predict(X_test)
# predicted up/down classes from logistic regression

log_prob = logistic_model.predict_proba(X_test)[:, 1]
# predicted probability copper goes up next month from logistic regression

rf_pred = rf_model.predict(X_test)
# predicted up/down classes from random forest

rf_prob = rf_model.predict_proba(X_test)[:, 1]
# predicted probability copper goes up next month from random forest

## Compare Models
Use simple metrics to decide which model is better.

In [ ]:
results_table = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y_test, log_pred),
        "roc_auc": roc_auc_score(y_test, log_prob)
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(y_test, rf_pred),
        "roc_auc": roc_auc_score(y_test, rf_prob)
    }
])
# build a small table comparing the models

results_table
# display the comparison

## Select Best Model
Choose the stronger model using ROC AUC.

In [ ]:
best_model_name = results_table.sort_values("roc_auc", ascending=False).iloc[0]["model"]
# sort models by ROC AUC and take the top one

if best_model_name == "Logistic Regression":
    best_model = logistic_model
    best_pred = log_pred
    best_prob = log_prob
else:
    best_model = rf_model
    best_pred = rf_pred
    best_prob = rf_prob
# save the chosen model and its predictions

print("Best model selected:", best_model_name)

## Build Test Results Table
Store the best model’s outputs alongside the test-period data.

In [ ]:
test_results = test_df.copy()
# start with the original test data

test_results["pred_up"] = best_pred
# predicted class: 1 for up, 0 for down

test_results["pred_prob_up"] = best_prob
# predicted probability copper goes up next month

test_results["actual_up"] = y_test
# actual next-month direction for comparison

test_results.head()
# preview the test results

## Estimate Typical Up and Down Moves
Use training data to estimate how big an up month or down month usually is.

In [ ]:
avg_up_return = train_df.loc[train_df[TARGET_COLUMN] == 1, RETURN_TARGET_COLUMN].mean()
# average next-month return during historical months where copper went up

avg_down_return = train_df.loc[train_df[TARGET_COLUMN] == 0, RETURN_TARGET_COLUMN].mean()
# average next-month return during historical months where copper went down

print("Average up return:", avg_up_return)
print("Average down return:", avg_down_return)

## Convert Direction Predictions into Price Predictions
Turn up/down predictions into estimated next-month copper prices.

In [ ]:
test_results["pred_next_ret_rule"] = np.where(
    test_results["pred_up"] == 1,     # if model predicts up
    avg_up_return,                    # use average historical up-month return
    avg_down_return                   # otherwise use average historical down-month return
)
# this gives a simple estimated return for each test row

test_results["pred_next_price_rule"] = test_results[PRICE_COLUMN] * (1 + test_results["pred_next_ret_rule"])
# convert predicted return into predicted next-month copper price

test_results["actual_next_price"] = test_results[PRICE_COLUMN] * (1 + test_results[RETURN_TARGET_COLUMN])
# compute the actual next-month copper price for comparison

test_results[[
    "copper_price",
    "pred_up",
    "pred_prob_up",
    "pred_next_ret_rule",
    "pred_next_price_rule",
    "actual_next_price"
]].head()

## Make Latest Prediction
Use the newest available data point to generate a current forecast.

In [ ]:
latest_row = model_df.iloc[[-1]].copy()
# take the most recent row as a one-row dataframe

latest_features = latest_row[FEATURE_COLUMNS]
# extract only the features used by the model

latest_price = latest_row[PRICE_COLUMN].iloc[0]
# current copper price

latest_prob_up = best_model.predict_proba(latest_features)[:, 1][0]
# predicted probability copper goes up next month

latest_pred_up = int(latest_prob_up >= 0.5)
# convert probability into a yes/no up prediction

latest_pred_ret = avg_up_return if latest_pred_up == 1 else avg_down_return
# assign a typical return size based on the predicted direction

latest_pred_price = latest_price * (1 + latest_pred_ret)
# convert predicted return into predicted next-month copper price

print("Latest date:", latest_row.index[0])
print("Current copper price:", latest_price)
print("Predicted probability up:", latest_prob_up)
print("Predicted direction:", "UP" if latest_pred_up == 1 else "DOWN")
print("Predicted next-month price:", latest_pred_price)

## Save Outputs
Write the prediction outputs to csv files for the Streamlit app.

In [ ]:
OUTPUT_DIR = Path("data")
# use the same data folder as the scraping notebook

OUTPUT_DIR.mkdir(exist_ok=True)
# create the folder if it does not already exist

PREDICTIONS_PATH = OUTPUT_DIR / "copper_model_predictions.csv"
LATEST_PATH = OUTPUT_DIR / "copper_latest_prediction.csv"
SUMMARY_PATH = OUTPUT_DIR / "copper_model_summary.csv"
# define output file paths

test_results[[
    "copper_price",
    "pred_up",
    "pred_prob_up",
    "pred_next_ret_rule",
    "pred_next_price_rule",
    "actual_next_price",
    "actual_up"
]].to_csv(PREDICTIONS_PATH)
# save the test-period prediction table

latest_prediction = pd.DataFrame([{
    "date": latest_row.index[0],
    "current_copper_price": latest_price,
    "predicted_prob_up": latest_prob_up,
    "predicted_direction": "UP" if latest_pred_up == 1 else "DOWN",
    "predicted_next_month_price": latest_pred_price,
    "best_model": best_model_name
}])
# make a one-row dataframe with the latest live prediction

latest_prediction.to_csv(LATEST_PATH, index=False)
# save the latest prediction

summary_df = pd.DataFrame([{
    "best_model": best_model_name,
    "accuracy": accuracy_score(y_test, best_pred),
    "roc_auc": roc_auc_score(y_test, best_prob)
}])
# keep a small summary table for the app

summary_df.to_csv(SUMMARY_PATH, index=False)
# save the model summary

print("Saved predictions to:", PREDICTIONS_PATH)
print("Saved latest prediction to:", LATEST_PATH)
print("Saved model summary to:", SUMMARY_PATH)

## Quick Preview
Preview the saved outputs.

In [ ]:
summary_df
# display the summary metrics

latest_prediction
# display the most recent forecast

test_results.tail()
# preview the most recent test-period rows